# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. We'll use the dataset's Croissant schema to review metadata, extract data from record sets, and perform exploratory analysis.

### Dataset Source
The dataset source is provided via its Croissant schema URL:

In [ ]:
# Ensure the required package is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset (this downloads and parses the Croissant JSON-LD)
dataset = mlc.Dataset(croissant_url)

# Display dataset metadata
metadata = dataset.metadata
print(f"{metadata.name}\n\n{metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

Let's list all record sets in the dataset, and for each, display its fields and unique `@id` identifiers. These IDs are essential for referencing data elements in subsequent operations.

In [ ]:
# List all record sets by their @id and gather field info
record_sets = dataset.record_sets
if len(record_sets) == 0:
    print('No top-level RecordSet definitions found in Croissant schema.')
else:
    for rs in record_sets:
        print(f"\nRecordSet: {rs.name}\n@id: {rs.id}")
        print("Fields and their @id's:")
        for field in rs.fields:
            print(f"  - {field.name} (@id: {field.id})")
        print("---")

If your dataset contains large or named record sets, you may wish to explore a preview of the records from a chosen record set. Replace `<record_set_id>` below with the `@id` you're interested in.

In [ ]:
# Example: Preview records from a specific record set by @id (replace the string with actual @id as discovered above)
example_record_set_id = None
if len(record_sets) > 0:
    example_record_set_id = record_sets[0].id

if example_record_set_id:
    print(f"\nPreviewing first 3 records from record set: {example_record_set_id}\n")
    for idx, rec in enumerate(dataset.records(record_set=example_record_set_id)):
        print(rec)
        if idx == 2:
            break
else:
    print("No record sets to preview.")

## 3. Data Extraction
Load data from each record set into pandas DataFrames for analysis, referencing each by its `@id`.

Below, we build a dictionary of DataFrames, keyed by each record set's `@id`, so you can access any record set's data easily.

In [ ]:
dataframes = {}
record_set_ids = [rs.id for rs in record_sets]
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Loaded record set: {rs_id}, rows: {len(df)}, columns: {df.columns.tolist()}")

# Print the columns of the first record set as an example
if record_set_ids:
    example_rs = record_set_ids[0]
    print(f"\nColumns in record set '@id': {example_rs}")
    print(dataframes[example_rs].columns.tolist())
    display(dataframes[example_rs].head())
else:
    print("No record sets available to display.")

## 4. Exploratory Data Analysis (EDA)
Let's perform some common analysis tasks using the record set and field `@id`s. We will:
- Filter records based on a numeric field referenced by its `@id`
- Normalize this field
- Optionally, group by a categorical field if available

_Please edit the variables `analyzed_rs_id`, `numeric_field_id`, and `group_field_id` below to fit your actual data fields._

In [ ]:
# Edit these variables to fit your dataset structure
# Use the actual @id of the record set and fields you want to analyze
if record_set_ids:
    analyzed_rs_id = record_set_ids[0]  # Example: use first record set
    df = dataframes[analyzed_rs_id]
    print(f"Analyzing record set: {analyzed_rs_id}")
    numeric_fields = df.select_dtypes(include=['number']).columns.tolist()
    print(f"Numeric fields detected: {numeric_fields}")

    if numeric_fields:
        numeric_field_id = numeric_fields[0]  # Example: pick first numeric
        threshold = df[numeric_field_id].median()  # Use median as an example threshold
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"\nFiltered records where {numeric_field_id} > {threshold}:")
        print(filtered_df[[numeric_field_id]].head())

        # Normalize
        filtered_df = filtered_df.copy()  # Avoid SettingWithCopyWarning
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nFirst 5 normalized values for {numeric_field_id}:")
        print(filtered_df[[numeric_field_id, norm_col]].head())

        # Try group by a categorical field, if exists
        group_fields = df.select_dtypes(include=['object', 'category']).columns.tolist()
        if group_fields:
            group_field_id = group_fields[0]
            grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
            print(f"\nGroup mean of {numeric_field_id} by {group_field_id} (top 5):")
            print(grouped.head())
    else:
        print("No numeric field detected for filtering and normalization.")
else:
    print("No record sets available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields within a selected record set.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Simple histogram of numeric field distribution
if record_set_ids and 'numeric_field_id' in locals() and numeric_field_id in df.columns:
    plt.figure(figsize=(7, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

# Boxplot by group field
if record_set_ids and 'group_field_id' in locals() and group_field_id in df.columns and numeric_field_id in df.columns:
    plt.figure(figsize=(8, 4))
    sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.xticks(rotation=30)
    plt.show()

## 6. Conclusion
We've demonstrated how to load, overview, extract, and explore data from a Croissant-formatted FAIR² dataset using the `mlcroissant` library. By referencing entities via `@id`, we've ensured unambiguous data access and processing. Continue your analysis with more detailed EDA and modeling as needed for your project.